<a href="https://colab.research.google.com/github/emmakelly7/A01-data-wrangling/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
temp_price = pd.to_numeric(
    df['price'].astype(str).str.replace('$', '', regex=False)
)

revenue_before_dupes = (temp_price * df['qty']).sum()

dupes = df.duplicated().sum()
print("Duplicate Rows:", dupes)

df = df.drop_duplicates().copy()

temp_price_after = pd.to_numeric(
    df['price'].astype(str).str.replace('$', '', regex=False)
)

revenue_after_dupes = (temp_price_after * df['qty']).sum()

log("duplicates", "Dropped rows with exact duplicates", dupes)

print("Revenue before duplicate removal:", revenue_before_dupes)
print("Revenue after duplicate removal:", revenue_after_dupes)
print("Duplicates remaining:", df.duplicated().sum())

Duplicate Rows: 15
[duplicates] Dropped rows with exact duplicates (15 row(s))
Revenue before duplicate removal: 4852.5
Revenue after duplicate removal: 4594.5
Duplicates remaining: 0


In order to drop duplicate rows, I used df.duplicated().sum() to find the number of duplicate rows. It output that there were 15 duplicated rows, which was consistent with the injected duplicates from df.sample(15, random_state=1). df = df.drop_duplicates().copy() was to get rid of the rows that were the same in all of the columns, and then the .copy made a new copy of the clean data. log("duplicates", "Dropped rows with exact duplicates", dupes) was to record the change with the 15 rows and then I checked the number of duplicates that were still in the dataset to make sure that the cleaning step worked.

### TODO 2 — clean `price` -> float

In [4]:
print(df['price'].value_counts())
print("Price type before:", df['price'].dtype)

price
6.0       67
$12.00    65
$7.50     59
7.5       58
24        51
Name: count, dtype: int64
Price type before: object


In [5]:
df['price'] = df['price'].astype(str)
df['price'] = df['price'].str.replace('$', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

print("Price type after:", df['price'].dtype)
print("Missing prices after conversion:", df['price'].isna().sum())

log("price", "Standardized and converted 300 price values to numeric", len(df))

print(df['price'].value_counts())

Price type after: float64
Missing prices after conversion: 0
[price] Standardized and converted 300 price values to numeric (300 row(s))
price
7.5     117
6.0      67
12.0     65
24.0     51
Name: count, dtype: int64


I first used print(df['price'].value_counts()) to see the price column and how the data was input, and then I used print("Price type before:", df['price'].dtype) to check the data. Price type before was classified as an object because there were some prices with dollar signs and some without. I converted the values to strings so that I could use .str.replace() to get rid of the dollar signs, and then used pd.to_numeric() with errors='coerce' to convert the cleaned values to numeric data. The price after was float and there were no missing prices after the conversion, so the values all converted in the right way. I used value_counts() again to print the prices and make sure that numbers like $7.50 and 7.5 were combined to a singular numeric value.

### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [6]:
print(df['qty'].value_counts(dropna=False))
print("Qty type before:", df['qty'].dtype)

qty
 1.0    159
 2.0     69
 3.0     47
-1.0     13
 NaN     12
Name: count, dtype: int64
Qty type before: float64


In [7]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

missing_qty = df['qty'].isna().sum()
negative_qty = (df['qty'] < 0).sum()

print("Missing qty:", missing_qty)
print("Negative qty:", negative_qty)

Missing qty: 12
Negative qty: 13


In [8]:
revenue_before_qty = (df['price'] * df['qty']).sum()

df = df[df['qty'].notna() & (df['qty'] > 0)].copy()

revenue_after_qty = (df['price'] * df['qty']).sum()

rows_removed = missing_qty + negative_qty
log("qty", "Dropped rows with missing or negative quantity", rows_removed)

print("Revenue before:", revenue_before_qty)
print("Revenue after:", revenue_after_qty)
print("Rows remaining:", len(df))
print("Missing qty remaining:", df['qty'].isna().sum())
print("Negative qty remaining:", (df['qty'] < 0).sum())

[qty] Dropped rows with missing or negative quantity (25 row(s))
Revenue before: 4594.5
Revenue after: 4740.0
Rows remaining: 275
Missing qty remaining: 0
Negative qty remaining: 0


I first used value_counts(dropna=False) to see all of the values in the qty column with the missing values, and then I checked the data type. The quantity column was already float64, but I used pd.to_numeric() with errors='coerce' to make sure all values were numeric and that any values that could not be converted would be listed as missing values. Then, I counted the missing and negative quantities before removing them and found 12 missing values and 13 negative values. I filtered the data to keep only rows where qty was not missing and was greater than zero, which changed revenue from 4,594.50 dollars to 4,740.00 dollars. Finally, I checked the data again and found 0 missing quantities and 0 negative quantities remaining to make sure I did it correctly.

### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [10]:
print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64


In [11]:
ITEM_MAP = {
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger'
}

df['item'] = df['item'].map(ITEM_MAP)
print(df['item'].value_counts())

log("item", "Fixed different spellings of items into 3 canonical products", 126)

item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
[item] Fixed different spellings of items into 3 canonical products (126 row(s))


I used value_counts() to see the item column and found six different item names that were meant to represent three products. “Foam Finger” and “foam finger,” “Rain Poncho” and “rain poncho,” and “Cheeseburger” and “cheese burger” represented the same products. ITEM_MAP mapped each to categorize them under a singular item name per singular product. Then, .map() applied it to the column. This changed the item name in 126 rows. Then, I used value_counts() again to check that only the three item names were being listed and that their counts still equaled 275 rows.

### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [12]:
print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64


In [13]:
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear',
    'Merch': 'Merch',
    'Apparel': 'Merch'
}

df['category'] = df['category'].map(CATEGORY_MAP)
print(df['category'].value_counts())

log("category", "Normalized categories and combined Apparel into Merch as a business decision", 132)

category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64
[category] Normalized categories and combined Apparel into Merch as a business decision (132 row(s))


I used value_counts() to see the category column, and it output six different category names. “Food” and “food” represented the same category, and “RainGear” and “rain-gear” represented the same category, so I standardized those values. Apparel and Merch were not just different spellings, so combining them required a business decision. I decided to categorize Apparel under Merch because apparel can be treated as merchandise for the purpose of reporting the categories in this dataset. I created CATEGORY_MAP to map the six category values into three standardized categories and used .map() to fix the column. This changed the category value in 132 rows. Then, I used value_counts() again to make sure that only Food, Merch, and RainGear remained and that their counts still equaled 275 rows.

### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [14]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Foam Finger', 'Rain Poncho', 'Cheeseburger'}
assert set(df['category'].unique()) == {'Food', 'Merch', 'RainGear'}
print('clean:', df.shape)

clean: (275, 5)


I used assertions to check that each of the TODO steps worked and cleaned the data. I checked that there were no duplicate rows, that all quantities were at least 1, and that price had been converted to a float. I also checked that the item column only had the three canonical  names and that the category column contained only Food, Merch, and RainGear.

### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [15]:
df['revenue'] = df['price'] * df['qty']
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)

print("Revenue by category:")
print(revenue_by_category)

total_revenue = df['revenue'].sum().round(2)
print("Overall revenue:", total_revenue)

Revenue by category:
category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Overall revenue: 4740.0


**What I would tell the vendor:** _..._

I would tell the vendor to stock more Food because it had the highest revenue by category when comparing against Merch and RainGear.

### TODO 8 — read back your log

In [16]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,Dropped rows with exact duplicates,15
1,price,Standardized and converted 300 price values to...,300
2,qty,Dropped rows with missing or negative quantity,25
3,item,Fixed different spellings of items into 3 cano...,126
4,category,Normalized categories and combined Apparel int...,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a. The cleaning step that changed my revenue total the most was removing duplicate rows. It was 4,852.50 dollars before and 4,594.50 dollars after.

b.) One decision I made where a reasonable person could have chosen differently was combining Apparel with Merch. The other choice would have been keeping Apparel as its own category because Apparel and Merch are different labels not just different spellings of the same category. Keeping them separate would not have changed the overall reported revenue of 4,740.00 dollars, but it would have changed the revenue reported by category because Apparel revenue would have been reported separately instead of being included under Merch. I chose to combine them because apparel can be categorized as merchandise, and I wanted to use more general and broader categories.